In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

print("Libraries Loaded")

Libraries Loaded


In [2]:
DATA_PATH = "../data/raw/boi_transactions.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

(9082, 3925)


In [3]:
df.drop(columns=["Unnamed: 0"], inplace=True)

print(df.shape)

(9082, 3924)


In [4]:
target = "F3924"

X = df.drop(columns=[target])

y = df[target]

print(X.shape)
print(y.shape)

(9082, 3923)
(9082,)


In [5]:
empty_columns = X.columns[X.isnull().all()]

print("Empty Columns:", len(empty_columns))

X = X.drop(columns=empty_columns)

print(X.shape)

Empty Columns: 63
(9082, 3860)


In [6]:
constant_columns = [
    col
    for col in X.columns
    if X[col].nunique(dropna=False) == 1
]

print(len(constant_columns))

X = X.drop(columns=constant_columns)

print(X.shape)

1
(9082, 3859)


In [7]:
missing_percentage = X.isnull().mean() * 100

high_missing = missing_percentage[
    missing_percentage > 80
].index

print("Columns Removed:", len(high_missing))

X = X.drop(columns=high_missing)

print(X.shape)

Columns Removed: 845
(9082, 3014)


In [8]:
leakage_features = ["F3912"]

X = X.drop(columns=leakage_features)

print(X.shape)

(9082, 3013)


In [9]:
numerical_columns = X.select_dtypes(include=["number"]).columns

categorical_columns = X.select_dtypes(include=["object"]).columns

print("Numerical:", len(numerical_columns))
print("Categorical:", len(categorical_columns))

Numerical: 3005
Categorical: 8


In [10]:
from sklearn.impute import SimpleImputer

# Numerical features
num_imputer = SimpleImputer(strategy="median")

# Categorical features
cat_imputer = SimpleImputer(strategy="most_frequent")

X[numerical_columns] = num_imputer.fit_transform(X[numerical_columns])

X[categorical_columns] = cat_imputer.fit_transform(X[categorical_columns])

print("Remaining Missing Values :", X.isnull().sum().sum())

Remaining Missing Values : 0


In [11]:
missing_after = X.isnull().sum()

missing_after[missing_after > 0]

Series([], dtype: int64)

In [12]:
categorical_columns

Index(['F2230', 'F3886', 'F3888', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893'], dtype='str')

In [13]:
for col in categorical_columns:
    print("=" * 50)
    print(col)
    print(X[col].value_counts().head(10))

F2230
F2230
Oct25    9001
Sep25      48
Nov25      23
Dec25      10
Name: count, dtype: int64
F3886
F3886
Savings         5956
Current         2051
MSME Micro       337
MSME Small       242
Staff Loans      108
Agri Adv          96
Term Deposit      87
MSME Medium       71
Corp Adv          23
Gold Loan         23
Name: count, dtype: int64
F3888
F3888
9-19-2025     24
9-17-2025     20
10-13-2025    18
9-23-2025     16
11-7-2025     16
9-1-2025      15
9-6-2025      15
9-22-2025     15
10-9-2025     15
9-15-2025     14
Name: count, dtype: int64
F3889
F3889
G365D    7544
L365D     397
L7D       386
L180D     313
L90D      207
L31D      148
L14D       87
Name: count, dtype: int64
F3890
F3890
M     2900
SU    2390
R     2015
U     1777
Name: count, dtype: int64
F3891
F3891
selfemployed    3951
salaried        1909
student         1185
agriculture     1112
housewife        660
others           169
retired           96
Name: count, dtype: int64
F3892
F3892
M    7605
F    1416
O      61
Name:

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training :", X_train.shape)
print("Testing  :", X_test.shape)

print("\nTarget Distribution")
print(y_train.value_counts())
print(y_test.value_counts())

Training : (7265, 3013)
Testing  : (1817, 3013)

Target Distribution
F3924
0    7200
1      65
Name: count, dtype: int64
F3924
0    1801
1      16
Name: count, dtype: int64


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Detect columns from TRAIN ONLY
numerical_columns = X_train.select_dtypes(include=["number"]).columns
categorical_columns = X_train.select_dtypes(include=["object"]).columns

print("Numerical :", len(numerical_columns))
print("Categorical :", len(categorical_columns))


# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_columns),
    ("cat", categorical_transformer, categorical_columns)
])

print("Preprocessor Ready!")

Numerical : 3005
Categorical : 8
Preprocessor Ready!


In [17]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(7265, 6883)
(1817, 6883)


In [18]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

print("Before SMOTE")
print(y_train.value_counts())

print("\nAfter SMOTE")
print(y_train_smote.value_counts())

print("\nShape :", X_train_smote.shape)

Before SMOTE
F3924
0    7200
1      65
Name: count, dtype: int64

After SMOTE
F3924
0    7200
1    7200
Name: count, dtype: int64

Shape : (14400, 6883)
